# レッスン 02 - Microsoft Agent Framework の探求

**Microsoft Agent Framework (MAF)** は、AIエージェントを構築するための統一フレームワークです。これは、4つのコアビルディングブロックから成るシンプルで構成可能なアーキテクチャを提供します:

- **Client** – AIモデルのエンドポイントに接続し、通信を処理する
- **Agent** – 指示やツール定義をもってクライアントをラップする
- **Tools** – モデルが呼び出せるカスタム関数でエージェントの機能を拡張する
- **Session** – マルチターンインタラクションのために会話履歴を管理する

このレッスンでは、これらの概念を使って、目的地の空席確認を行う<strong>旅行予約エージェント</strong>を構築します。


## セットアップ


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## エージェントフレームワークアーキテクチャの理解

Microsoft Agent Frameworkは階層化されたアーキテクチャに従っています：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. <strong>クライアント</strong> – `FoundryChatClient` はAzure OpenAIのデプロイに接続します。認証、リクエストのフォーマット、レスポンスの解析を処理します。
2. <strong>エージェント</strong> – クライアントから `provider.create_agent()` を通じて作成され、モデルへのアクセス、指示（システムプロンプト）、ツールを組み合わせます。
3. <strong>ツール</strong> – `@tool` デコレータが付いたPython関数で、エージェントがアクションを実行したりデータを取得したりするために呼び出すことができます。
4. <strong>セッション</strong> – 会話履歴を保存する `AgentSession` オブジェクト（`agent.create_session()` により作成）で、エージェントが以前のコンテキストを覚えて多ターンダイアログを可能にします。

各層を段階的に構築していきましょう。


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## @tool デコレーターを使ったツールの追加

ツールはエージェントがテキスト生成以外のアクションを実行できるようにします。`@tool` デコレーターは通常のPython関数をエージェントが呼び出せるものに変換します。

主なポイント:
- モデルが各パラメーターを理解できるように、`Annotated[type, "description"]` を使用します。
- ドックストリングはモデルが見るツールの説明になります。
- `approval_mode="never_require"` はユーザーの確認なしにツールが自動的に実行されることを意味します。


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ツールを使用したエージェントの作成

ここで、クライアント、指示、およびツールをエージェントに統合します。`instructions` はシステムプロンプトとして機能し、エージェントのペルソナと行動を定義します。


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## セッションを使ったマルチターン会話

`AgentSession`（`agent.create_session()`で作成） は会話中のすべてのメッセージを追跡します。同じセッションを各 `agent.run()` 呼び出しに渡すことで、エージェントは会話の全履歴にアクセスでき、以前のメッセージを参照できます。

各ターンでエージェントが利用可能性チェッカーを呼び出せるように、`tools=[check_destination_availability]` を渡しています。


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## 概要

このレッスンでは、Microsoft Agent Frameworkの4つの柱について学びました：

| 概念 | 学んだこと |
|---------|------------------|
| <strong>クライアント</strong> | `FoundryChatClient` は資格情報ベースの認証でAzure OpenAIに接続します |
| <strong>エージェント</strong> | `provider.create_agent()` はモデル接続に指示と名前をまとめます |
| <strong>ツール</strong> | `@tool` デコレーターはエージェントが呼び出すPython関数を公開します |
| <strong>セッション</strong> | `agent.create_session()` は複数ターンにわたる会話履歴を維持します |

これらの構成要素は組み合わさって、自然な会話を行い、外部関数を呼び出し、文脈を維持できるエージェントを作成します。これは後のレッスンでより高度なエージェントパターンの基礎となります。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
